In [54]:
11

11

In [ ]:
import os
import random
from tqdm import tqdm
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Any
import sentencepiece as spm

import evaluate
import numpy as np
import torch
from datasets import load_dataset, DatasetDict, Dataset, concatenate_datasets, disable_progress_bar
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed
)
from transformers.trainer_utils import get_last_checkpoint
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score

In [56]:
# Replace with your HuggingFace dataset name
DATASET_PATH = "projected_datasets"
# Adjust these if your dataset has different column names
SOURCE_COLUMN = "shp"  # or "shipibo"
TARGET_COLUMN = "spa"   # or "spanish"

BASE_MODEL_NAME = "xlm-roberta-base"
BASE_MODEL_LOCAL_PATH = "models/xlm-roberta-base"
BASE_TOKENIZER_NAME = "xlm-roberta-base"
BASE_TOKENIZER_LOCAL_PATH = "tokenizers/xlm-roberta-base"  # Assuming tokenizer is saved with the model

TRAINED_MODEL_NAME = "bert-sentiment-shipibo"
TRAINED_MODEL_OUTPUT_DIR = f"models/bert-sentiment-shipibo"
TRAINED_MODEL_LOGS_DIR = f"logs/bert-sentiment-shipibo"

CONFIDENCE_THRESHOLD = 0.6
MAX_LENGTH = 128

In [57]:
def clear_gpu_memory():
    """Clear GPU memory cache"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        import gc
        gc.collect()
        print("🧹 GPU memory cleared")

In [58]:
print(f"📊 Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

📊 Device: GPU
   GPU: NVIDIA GeForce RTX 4070 Ti SUPER
   Memory: 17.17 GB


In [ ]:
def load_dataset(dataset_path: str) -> DatasetDict:
    """
    Load Shipibo-Spanish parallel corpus with Sentiment labels from device
    
    Args:
        dataset_path: Path to the dataset file (e.g., "data/train.parquet")

    Returns:
        DatasetDict with train/validation/test splits
    """
    print(f"\n📥 Loading dataset: {dataset_path}")

    # Load the dataset
    dataset = DatasetDict()
    dataset["train"] = Dataset.from_parquet(os.path.join(dataset_path, "train-robertuito.parquet"))
    dataset["validation"] = Dataset.from_parquet(os.path.join(dataset_path, "validation-reviewed.parquet"))

    # Display statistics
    print("\n📊 Dataset Statistics:")
    for split in dataset.keys():
        print(f"   {split}: {len(dataset[split])} examples")
        if len(dataset[split]) > 0:
            # Show first example
            example = dataset[split][0]
            print(f"   Example Shipibo: {example.get('shp', example.get('shipibo', 'N/A'))[:50]}...")
            print(f"   Example Spanish: {example.get('spa', example.get('spanish', 'N/A'))[:50]}...")
    
    return dataset


In [60]:
full_dataset = load_dataset(DATASET_PATH)


📥 Loading dataset: projected_datasets

📊 Dataset Statistics:
   train: 16505 examples
   Example Shipibo: Jato shinamawe mesko yokabo axon neskaakin....
   Example Spanish: Ahora hazles recordar a través de diferentes pregu...
   validation: 2045 examples
   Example Shipibo: Metsara iwanke....
   Example Spanish: Fue maravilloso....
   test: 2075 examples
   Example Shipibo: Kirikanin non raoki ika rabiti wishaxon axetixobon...
   Example Spanish: Escribe en un cuaderno un poema a nuestras plantas...


In [61]:
def prepare_tokenizer(tokenizer_path: str, source_key: str, target_key: str) -> AutoTokenizer:
    """
    Load and configure tokenizer for Shipibo→Spanish
    """
    print("\n🔧 Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

    return tokenizer

In [62]:
def prepare_sentiment_model(model_name: str, model_path: str, num_labels: int):
    """
    Get the pre-trained model locally.
    
    Args:
        model_name: Name of the pre-trained model to download.
        model_path: Directory where the model is saved.
    """
    print(f"\n⬇️  Loading model: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=num_labels, ignore_mismatched_sizes=True)
    model.config.label2id = {'NEG': 0, 'NEU': 1, 'POS': 2}
    model.config.id2label = {0: 'NEG', 1: 'NEU', 2: 'POS'}
    print(f"   Model loaded from: {model_path}")
    return model
    

In [63]:
def download_sentiment_model(model_name: str, output_dir: str):
    """
    Download and save the pre-trained model locally.
    
    Args:
        model_name: Name of the pre-trained model to download.
        output_dir: Directory to save the downloaded model.
    """
    print(f"\n⬇️  Downloading model: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.save_pretrained(output_dir)
    print(f"   Model saved to: {output_dir}")

In [64]:
def download_tokenizer(tokenizer_name: str, output_dir: str):
    """
    Download and save the tokenizer locally.
    
    Args:
        tokenizer_name: Name of the tokenizer to download.
        output_dir: Directory to save the downloaded tokenizer.
    """
    print(f"\n⬇️  Downloading tokenizer: {tokenizer_name}")
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    tokenizer.save_pretrained(output_dir)
    print(f"   Tokenizer saved to: {output_dir}")

## Fine-Tune Sentiment Model

In [65]:
# Download Base model and tokenizer if not already present
if not os.path.exists(BASE_MODEL_LOCAL_PATH):
    download_sentiment_model(BASE_MODEL_NAME, BASE_MODEL_LOCAL_PATH)
base_model = prepare_sentiment_model(BASE_MODEL_NAME, BASE_MODEL_LOCAL_PATH, 3)
if not os.path.exists(BASE_TOKENIZER_LOCAL_PATH):
    download_tokenizer(BASE_TOKENIZER_NAME, BASE_TOKENIZER_LOCAL_PATH)
base_tokenizer = prepare_tokenizer(BASE_TOKENIZER_NAME, SOURCE_COLUMN, TARGET_COLUMN)


⬇️  Loading model: xlm-roberta-base


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at models/xlm-roberta-base and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   Model loaded from: models/xlm-roberta-base

🔧 Loading tokenizer...


In [ ]:
def preprocess_function(dataset: Dataset, tokenizer):
    """Tokenize and encode the Shipibo texts."""
    encodings = tokenizer(dataset["shp"], truncation=True, padding=False, max_length=256)
    # Map sentiment labels to IDs
    label_map = {'NEG': 0, 'NEU': 1, 'POS': 2}
    encodings['sentiment_label'] = [label_map[label] for label in dataset['sentiment_label']]
    return encodings

In [ ]:
def compute_metrics(pred):
    preds = np.argmax(pred.predictions, axis=1)
    labels = pred.label_ids

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="weighted"),
        "recall": recall_score(labels, preds, average="weighted"),
        "f1": f1_score(labels, preds, average="weighted"),
    }

In [68]:
# Preprocess datasets
train_dataset = full_dataset["train"].map(preprocess_function, fn_kwargs={"tokenizer": base_tokenizer}, batched=True)
test_dataset = full_dataset["test"].map(preprocess_function, fn_kwargs={"tokenizer": base_tokenizer}, batched=True)
validation_dataset = full_dataset["validation"].map(preprocess_function, fn_kwargs={"tokenizer": base_tokenizer}, batched=True)


# Format datasets for PyTorch
train_dataset = train_dataset.rename_column("sentiment_label", "label")
train_dataset = train_dataset.remove_columns(["shp", "spa", "__index_level_0__","sentiment_score"])

test_dataset = test_dataset.rename_column("sentiment_label", "label")
test_dataset = test_dataset.remove_columns(["shp", "spa", "__index_level_0__","sentiment_score"])

validation_dataset = validation_dataset.rename_column("sentiment_label", "label")
validation_dataset = validation_dataset.remove_columns(["shp", "spa", "__index_level_0__","sentiment_score"])

Map:   0%|          | 0/16505 [00:00<?, ? examples/s]

Map:   0%|          | 0/2075 [00:00<?, ? examples/s]

Map:   0%|          | 0/2045 [00:00<?, ? examples/s]

In [69]:
# Training hyperparameters
BATCH_SIZE = 4           # Adjust based on GPU memory
LEARNING_RATE = 3e-5
NUM_EPOCHS = 10
WARMUP_STEPS = 500
SEED = 42

In [70]:
# Training arguments
training_args = TrainingArguments(
    output_dir=TRAINED_MODEL_OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=WARMUP_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    warmup_ratio=0.1,
    logging_steps=50,
    logging_dir=TRAINED_MODEL_LOGS_DIR,
    report_to="none",  # Disable wandb/tensorboard
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available,
    remove_unused_columns=False,  # Important for custom collator
)

In [71]:
trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    tokenizer=base_tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_7661/1427421482.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [72]:
clear_gpu_memory()

🧹 GPU memory cleared


In [73]:
# Train
print(f"\n{'='*60}")
print("Starting training...")
print(f"{'='*60}\n")

clear_gpu_memory()

trainer.train()

# Save final model
print(f"\nSaving model to {TRAINED_MODEL_OUTPUT_DIR}...")
trainer.save_model(TRAINED_MODEL_OUTPUT_DIR)
trainer.tokenizer.save_pretrained(TRAINED_MODEL_OUTPUT_DIR)
print("\n✓ Training complete!")

clear_gpu_memory()


Starting training...

🧹 GPU memory cleared


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.713300,0.692977,0.757457,0.736736,0.623512,0.644744
2,0.714900,0.703400,0.792665,0.768456,0.687079,0.711007
3,0.448700,0.796300,0.820049,0.794795,0.733024,0.757122
4,0.583200,0.677148,0.832763,0.821394,0.731976,0.758481
5,0.474100,0.734426,0.849878,0.813303,0.780777,0.791682
6,0.424400,0.583304,0.880685,0.853872,0.824932,0.835847
7,0.219400,0.615528,0.889976,0.872806,0.833607,0.850991
8,0.321300,0.535671,0.905623,0.873537,0.871820,0.872645
9,0.205700,0.518752,0.917359,0.895825,0.889503,0.892570
10,0.251400,0.528178,0.917848,0.897202,0.887515,0.891951



Saving model to models/bert-sentiment-shipibo...


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.



✓ Training complete!
🧹 GPU memory cleared
